# Task B Run 30 — Gemma prompt ensemble on the fixed holdout

This is an evaluation-only experiment. It trains four matched Gemma-4-12B QLoRA classifiers on the fixed 530-row Task B holdout split:

| model | prompt | seeds |
|---|---|---|
| short-42, short-43 | the standard label-name prompt | 42, 43 |
| defs-42, defs-43 | the organisers' category-definition prompt | 42, 43 |

The final prediction is the arithmetic mean of all four probability matrices. We compare it with the two-seed standard-prompt mean using a paired bootstrap on exactly the same rows. This tests whether the prompt variants make useful, complementary errors; it does **not** produce a submission ZIP.

Each model uses the established Gemma-4-12B recipe: 4-bit QLoRA, rank 16, attention + MLP LoRA targets, 3 epochs, learning rate \`1e-4\`, balanced class weights, and the 320-token cap.

**Kaggle settings:** Internet on, GPU T4 x2, and a Kaggle secret named \`HF_TOKEN\` if Hugging Face asks for authentication. Save Version → Save & Run All. Expected time is roughly 3–4 hours.


In [ ]:
import json, os, pathlib, re, subprocess, sys, time

T_START = time.time()
HARD_STOP = T_START + 11 * 3600  # leave time for result files before Kaggle's limit
REPO = 'https://github.com/robinpnalex/Hastika-ICON2026.git'
BRANCH = 'task-b'
WORK = '/tmp/hastika'

def git(*args):
    return subprocess.run(['git', '-C', WORK, *args], check=True, capture_output=True, text=True).stdout.strip()

try:
    remote = subprocess.run(['git', 'ls-remote', '--exit-code', REPO, f'refs/heads/{BRANCH}'],
                            check=True, capture_output=True, text=True, timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit('Cannot reach GitHub. Turn on Settings -> Internet, then restart.') from e

if os.path.isdir(WORK + '/.git'):
    git('fetch', '-q', '--depth', '1', 'origin', BRANCH)
    git('reset', '-q', '--hard', 'FETCH_HEAD')
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, '--depth', '1', REPO, WORK], check=True)
head = git('rev-parse', 'HEAD')
assert head == remote, f'clone is at {head[:8]} but {BRANCH} is at {remote[:8]}'
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))

# Install the small preprocessing/runtime dependencies before importing repo modules.
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
print('HF_TOKEN available:', bool(os.environ.get('HF_TOKEN')))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

assert torch.cuda.is_available(), 'select GPU T4 x2'
N_GPU = torch.cuda.device_count()
print('repo:', git('log', '-1', '--oneline'))
print('gpus:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])

OUT = pathlib.Path('/kaggle/working/b30_prompt_ensemble_outputs')
RUNS, LOGS = OUT / 'runs', OUT / 'logs'
for p in (RUNS, LOGS):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = '/tmp/hf'
os.environ['HF_HUB_CACHE'] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import fingerprint, paired_bootstrap

MODEL = 'google/gemma-4-12B'
SPLIT = pathlib.Path(WORK) / 'data/derived/task_b_combined_holdout'
LABELS = ['Gender', 'Geo-political', 'Others', 'Political', 'Religion', 'Violence']
LABEL_COL = 'Hate Category'
train = pd.read_csv(SPLIT / 'train.csv')
holdout = pd.read_csv(SPLIT / 'holdout.csv')
assert (len(train), len(holdout)) == (3002, 530), (len(train), len(holdout))
HOLDOUT_FP = fingerprint(holdout['id'])
assert HOLDOUT_FP == 'f85f4f049b', HOLDOUT_FP
y_ho = holdout[LABEL_COL].map(LABELS.index).to_numpy()
print(f'train {len(train)}, holdout {len(holdout)}, fingerprint {HOLDOUT_FP}')

def report(y, probs, name):
    pred = probs.argmax(1)
    per = f1_score(y, pred, average=None, labels=range(len(LABELS)), zero_division=0)
    macro = float(f1_score(y, pred, average='macro', zero_division=0))
    acc = float(accuracy_score(y, pred))
    print(f'{name:32s} macro-F1 {macro:.4f}  acc {acc:.4f}  | ' +
          '  '.join(f'{c[:5]} {f:.3f}' for c, f in zip(LABELS, per)))
    return {'macro_f1': macro, 'accuracy': acc, 'per_class_f1': per.tolist()}

def compare(name, a, b):
    r = paired_bootstrap(y_ho, a, b)
    print(f'{name:32s} diff {r["diff"]:+.4f}  95% CI '
          f'[{r["ci95"][0]:+.4f}, {r["ci95"][1]:+.4f}]  P(better) {r["p_better"]:.2f}')
    return r


In [ ]:
# Keep the Gemma stack separate from Kaggle's system transformers/peft versions.
LLM_PY = '/tmp/llmenv/bin/python'
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip',
                    '/tmp/llmenv'], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, '-c',
                "import transformers, peft, bitsandbytes; print('LLM env:', transformers.__version__, peft.__version__)"],
               check=True)

def download_model():
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL, cache_dir=HF_CACHE, token=os.environ.get('HF_TOKEN') or None,
                      allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt',
                                      'tokenizer*', '*.jinja'])

def llm_job(prompt, seed):
    name = f'{prompt}_s{seed}'
    cmd = (f'{LLM_PY} -u -m hastika.task_b.llm_classifier --task b --model {MODEL} '
           f'--prompt {prompt} --train {SPLIT / "train.csv"} '
           f'--eval {SPLIT / "holdout.csv"} --predict {SPLIT / "holdout.csv"} '
           f'--out {RUNS / name} --seed {seed} --epochs 3 --lr 1e-4 --r 16 '
           f'--bs 8 --grad-accum 2 --eval-bs 16 --max-len 320 --cache {HF_CACHE}')
    return {'name': name, 'cmd': cmd, 'log': str(LOGS / f'{name}.log')}

def run_jobs(jobs):
    # A collapsed first seed is retried with a deterministic alternate seed. OOM is
    # retried at the same effective batch size with smaller micro-batches.
    registry = {j['name']: j for j in jobs}
    retried = set()

    def on_done(name, code):
        job = registry[name]
        if not code and name not in retried:
            return []
        log_path = pathlib.Path(job['log'])
        log = log_path.read_text(errors='replace') if log_path.exists() else ''
        extra = ''
        if code == 4 and name not in retried:
            m = re.search(r'--seed (\d+)', job['cmd'])
            if m:
                extra = f' --seed {int(m.group(1)) + 1000}'
                print(f'{name}: collapse detected; retrying with alternate seed')
        elif 'OutOfMemoryError' in log and name not in retried:
            extra = ' --bs 4 --grad-accum 4 --eval-bs 8'
            print(f'{name}: OOM detected; retrying with smaller micro-batches')
        if extra:
            retried.add(name)
            retry = dict(job)
            retry['cmd'] = job['cmd'] + extra
            registry[name] = retry
            return [retry]
        return []

    return run_queue(jobs, N_GPU, raise_on_fail=False, on_done=on_done, stop_at=HARD_STOP)

download_model()
jobs = [llm_job(prompt, seed) for prompt in ('short', 'defs') for seed in (42, 43)]
print('queued:', [j['name'] for j in jobs])


In [ ]:
codes = run_jobs(jobs)
print('exit codes:', codes)
print(f'elapsed {(time.time() - T_START) / 3600:.1f} h')


In [ ]:
def load_probs(name):
    run = RUNS / name
    p = run / 'holdout_probs.npy'
    ids = run / 'holdout_ids.csv'
    if not p.exists() or not ids.exists():
        return None
    got_ids = pd.read_csv(ids)['id'].tolist()
    if got_ids != holdout['id'].tolist():
        raise ValueError(f'{name} predictions do not match the fixed holdout')
    probs = np.load(p)
    assert probs.shape == (len(holdout), len(LABELS)), (name, probs.shape)
    return probs

models = {}
for prompt in ('short', 'defs'):
    for seed in (42, 43):
        name = f'{prompt}_s{seed}'
        p = load_probs(name)
        if p is None:
            raise RuntimeError(f'{name} did not finish; inspect {LOGS / (name + ".log")}')
        models[name] = p
        report(y_ho, p, name)

short_mean = np.mean([models['short_s42'], models['short_s43']], axis=0)
defs_mean = np.mean([models['defs_s42'], models['defs_s43']], axis=0)
ensemble = np.mean([short_mean, defs_mean], axis=0)
print()
scores = {
    'short_2seed_mean': report(y_ho, short_mean, 'short prompt, 2 seeds'),
    'defs_2seed_mean': report(y_ho, defs_mean, 'definitions prompt, 2 seeds'),
    'prompt_ensemble_4models': report(y_ho, ensemble, 'prompt ensemble, 4 models'),
}
print()
bootstrap = {
    'defs_minus_short': compare('defs mean - short mean', defs_mean, short_mean),
    'ensemble_minus_short': compare('ensemble - short mean', ensemble, short_mean),
}

np.save(OUT / 'short_mean_holdout_probs.npy', short_mean)
np.save(OUT / 'defs_mean_holdout_probs.npy', defs_mean)
np.save(OUT / 'prompt_ensemble_holdout_probs.npy', ensemble)
result = {
    'experiment': 'task_b_run30_gemma_prompt_ensemble_holdout',
    'commit': head,
    'holdout_rows': len(holdout),
    'holdout_fingerprint': HOLDOUT_FP,
    'models': list(models),
    'scores': scores,
    'bootstrap': bootstrap,
    'exit_codes': codes,
    'elapsed_hours': (time.time() - T_START) / 3600,
}
json.dump(result, open(OUT / 'result.json', 'w'), indent=2)
print()
print('Saved holdout probability matrices and result.json under', OUT)
print('No submission ZIP is created: use this result only to decide whether the prompt blend belongs in the final full-data recipe.')
